# 🤝 Xiangqi-R1 Federated Community Training (1-Click Google Colab T4 Free Tier)
### Đóng Góp Sức Mạnh GPU Colab T4 Miễn Phí Để Huấn Luyện Mô Hình AI Cờ Tướng Thế Hệ Mới Xiangqi-R1!

Chào mừng bạn đến với chiến dịch **Hợp Nhất Trí Tuệ AI Cờ Tướng Cộng Đồng (Federated Xiangqi-R1)**!
Bạn chỉ cần chạy 1-Click notebook này trên Google Colab T4 (Miễn phí 100%).
Mô hình sẽ tự động chạy 100-300 bước GRPO và tải tệp `adapter_model.safetensors` đóng góp của bạn lên HuggingFace Hub để hệ thống gộp thành mô hình đỉnh cao nhất!

- **Dataset Hub**: [hoduyquocbao/xiangqi-r1-dataset](https://huggingface.co/datasets/hoduyquocbao/xiangqi-r1-dataset)
- **Target Model**: [hoduyquocbao/xiangqi-r1-0.5b](https://huggingface.co/hoduyquocbao/xiangqi-r1-0.5b)

> ⚠️ **BẮT BUỘC**: Cấu hình HuggingFace Token trong Colab Secrets (🔑 icon bên trái) với key `HF_TOKEN`

In [ ]:
# 1. Khai Báo GPU & Cài Đặt Thư Viện Tối Ưu Unsloth
import os, sys, torch
print('⚡ GPU Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Đọc token từ Colab Secrets (KHÔNG hardcode trong mã nguồn)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    print('❌ HF_TOKEN chưa được cấu hình!')
    print('   → Vào 🔑 Secrets (bên trái) → Thêm key HF_TOKEN với giá trị write token của bạn')
    print('   → Tạo token tại: https://huggingface.co/settings/tokens')
else:
    print(f'✅ HF_TOKEN đã sẵn sàng ({HF_TOKEN[:8]}...)')

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub safetensors

In [ ]:
# 2. Tải Dataset Mới Nhất & Cấu Hình Huấn Luyện GRPO 100-300 Steps
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig

MODEL_ID = "hoduyquocbao/xiangqi-r1-0.5b"
DATASET_ID = "hoduyquocbao/xiangqi-r1-dataset"

print(f"📥 Đang tải dataset từ HuggingFace: {DATASET_ID}...")
ds = load_dataset(DATASET_ID, split="train")
print(f"✅ Đã nạp thành công {len(ds):,} mẫu cờ R1 tư duy 3-in-1!")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none"
)
print("🚀 Mô hình Unsloth LoRA Adapter đã sẵn sàng huấn luyện!")

In [ ]:
# 3. Reward Functions & Huấn Luyện 150 Steps GRPO (Tích hợp Resume & Rule Validation)
import os, re, json as _json

def reward_format(completions, **kwargs):
    """Thưởng completion có chứa JSON hợp lệ với key 'bestmove'."""
    rewards = []
    for text in completions:
        score = 0.0
        try:
            match = re.search(r'\{[^}]+\}', text)
            if match:
                obj = _json.loads(match.group())
                if 'bestmove' in obj:
                    score += 0.5
                    move = obj['bestmove']
                    if isinstance(move, str) and re.match(r'^[a-i][0-9][a-i][0-9]$', move):
                        score += 0.5
        except Exception:
            pass
        rewards.append(score)
    return rewards

def reward_rule(prompts, completions, **kwargs):
    """Thưởng completion có nước đi hợp lệ theo luật cờ Tướng bàn 9x10."""
    rewards = []
    for text in completions:
        score = 0.0
        try:
            match = re.search(r'\{[^}]+\}', text)
            if match:
                obj = _json.loads(match.group())
                move = obj.get('bestmove', '')
                if isinstance(move, str) and re.match(r'^[a-i][0-9][a-i][0-9]$', move):
                    # Kiểm tra ô xuất phát != ô đích
                    if move[:2] != move[2:]:
                        score += 0.5
                    # Kiểm tra tọa độ nằm trong bàn cờ 9x10
                    fc, fr = ord(move[0]) - ord('a'), int(move[1])
                    tc, tr = ord(move[2]) - ord('a'), int(move[3])
                    if 0 <= fc <= 8 and 0 <= fr <= 9 and 0 <= tc <= 8 and 0 <= tr <= 9:
                        score += 0.5
        except Exception:
            pass
        rewards.append(score)
    return rewards

def reward_thought(completions, **kwargs):
    """Thưởng completion có thẻ <thought> dài và chi tiết."""
    rewards = []
    for text in completions:
        score = 0.0
        if '<thought>' in text and '</thought>' in text:
            thought = text.split('<thought>')[1].split('</thought>')[0]
            score = min(len(thought) / 200.0, 1.0)
        rewards.append(score)
    return rewards

output_dir = "output_community"
args = GRPOConfig(
    output_dir=output_dir,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=10,
    max_steps=150,
    save_steps=50,
    max_prompt_length=512,
    max_completion_length=256,
    num_generations=4,
    report_to="none"
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_format, reward_thought, reward_rule],
    args=args,
    train_dataset=ds
)

# Kiểm tra & Tự động khôi phục từ Checkpoint nếu có
resume_checkpoint = None
if os.path.exists(output_dir):
    checkpoints = sorted([d for d in os.listdir(output_dir) if d.startswith('checkpoint-')])
    if checkpoints:
        resume_checkpoint = os.path.join(output_dir, checkpoints[-1])
        print(f"🔄 Tự động khôi phục huấn luyện từ Checkpoint: {resume_checkpoint}")

print("🔥 Bắt đầu phiên huấn luyện cộng đồng Colab T4 (150 Steps GRPO)...")
trainer.train(resume_from_checkpoint=resume_checkpoint)

model.save_lora("community_adapter")
print("💾 Đã xuất tệp adapter_model.safetensors thành công tại community_adapter/!")


In [ ]:
# 4. Upload Adapter Lên HuggingFace Hub
from huggingface_hub import HfApi
import time

if not HF_TOKEN:
    print('❌ Không thể upload — HF_TOKEN chưa được cấu hình!')
else:
    api = HfApi()
    stamp = int(time.time())
    adapter_path = 'community_adapter/adapter_model.safetensors'
    if os.path.exists(adapter_path):
        repo_path = f'community/adapter_{stamp}.safetensors'
        api.upload_file(
            path_or_fileobj=adapter_path,
            path_in_repo=repo_path,
            repo_id=DATASET_ID,
            repo_type='dataset',
            token=HF_TOKEN
        )
        print(f'✅ Đã upload adapter lên HuggingFace Hub: {repo_path}')
    else:
        print('❌ Không tìm thấy adapter_model.safetensors — huấn luyện có thể đã thất bại')